# Step 3.1: Timestamp Preprocessing

**Project:** AI for IoT — Smart Energy Consumption Prediction  
**Dataset:** Real PZEM + ESP32 Energy-Metering Dataset (`MODERATE_energy_metering_lights.csv`)  
**Task:** Convert ISO 8601 string timestamps (with `+01:00` / `+02:00` offsets) to timezone-aware UTC `datetime64[ns, UTC]`, rename column to `timestamp`, sort chronologically, and save intermediate dataset to `AI_IoT_Project/data/processed/lights_timestamp_processed.csv`.

In [1]:
# 1. Load protected original dataset
import pandas as pd
import numpy as np
import os

orig_csv_path = "../data/original/MODERATE_energy_metering_lights.csv"
proc_csv_path = "../data/processed/lights_timestamp_processed.csv"

df_raw = pd.read_csv(orig_csv_path)
print(f"Original Dataset Loaded. Shape: {df_raw.shape}")
print(f"Original Columns: {list(df_raw.columns)}")

Original Dataset Loaded. Shape: (830266, 7)
Original Columns: ['ISO 8601 timestamp (YYYY-MM-DD HH:MM:SS.ssssss)', 'voltage (V rms)', 'current (A rms)', 'energy (Wh)', 'frequency (Hz)', 'pf', 'power (W)']


In [2]:
# 2. Parse and convert timestamps to UTC
ts_col_orig = 'ISO 8601 timestamp (YYYY-MM-DD HH:MM:SS.ssssss)'

df_proc = df_raw.copy()
df_proc['timestamp'] = pd.to_datetime(df_proc[ts_col_orig], format='ISO8601', utc=True)
df_proc = df_proc.drop(columns=[ts_col_orig])

# Reorder columns putting 'timestamp' first
columns_order = ['timestamp', 'voltage (V rms)', 'current (A rms)', 'energy (Wh)', 'frequency (Hz)', 'pf', 'power (W)']
df_proc = df_proc[columns_order]

# Sort chronologically
df_proc = df_proc.sort_values(by='timestamp').reset_index(drop=True)

print("Timestamp Conversion Complete.")
print(f"New Dataframe Shape: {df_proc.shape}")
print(f"New Data Type: {df_proc['timestamp'].dtype}")

Timestamp Conversion Complete.
New Dataframe Shape: (830266, 7)
New Data Type: datetime64[us, UTC]


In [3]:
# 3. Verify timestamp properties and interval statistics
print("--- TIMESTAMP VERIFICATION ---")
print(f"Row count: {len(df_proc):,}")
print(f"Column count: {len(df_proc.columns)}")
print(f"Earliest Timestamp (UTC): {df_proc['timestamp'].min()}")
print(f"Latest Timestamp (UTC): {df_proc['timestamp'].max()}")
print(f"Is Timezone Aware: {df_proc['timestamp'].dt.tz is not None}")
print(f"Is Monotonically Increasing: {df_proc['timestamp'].is_monotonic_increasing}")
print(f"Duplicate Timestamps: {df_proc['timestamp'].duplicated().sum()}")

# Calculate sampling intervals
diffs = df_proc['timestamp'].diff().dt.total_seconds().dropna()
print("\n--- SAMPLING INTERVAL STATISTICS ---")
print(f"Minimum Interval: {diffs.min():.6f} seconds")
print(f"Maximum Interval: {diffs.max():.6f} seconds ({diffs.max()/3600:.2f} hours)")
print(f"Mean Interval:    {diffs.mean():.6f} seconds")
print(f"Median Interval:  {diffs.median():.6f} seconds")
print(f"Mode Interval:    {diffs.mode()[0]:.6f} seconds")

--- TIMESTAMP VERIFICATION ---
Row count: 830,266
Column count: 7
Earliest Timestamp (UTC): 2026-01-25 03:01:02.355659+00:00
Latest Timestamp (UTC): 2026-05-03 00:59:57.068040+00:00
Is Timezone Aware: True
Is Monotonically Increasing: True
Duplicate Timestamps: 0

--- SAMPLING INTERVAL STATISTICS ---
Minimum Interval: 0.158307 seconds
Maximum Interval: 11469.011039 seconds (3.19 hours)
Mean Interval:    10.189439 seconds
Median Interval:  10.010640 seconds
Mode Interval:    10.011274 seconds


In [4]:
# 4. Save processed dataset and confirm original file safety
df_proc.to_csv(proc_csv_path, index=False)
print(f"Processed file saved successfully to: {proc_csv_path}")

orig_exists = os.path.exists(orig_csv_path)
orig_size = os.path.getsize(orig_csv_path)
print(f"Original CSV exists: {orig_exists} (Size: {orig_size:,} bytes)")

Processed file saved successfully to: c:\Users\Tanya\OneDrive\Desktop\AI-for-IoT-Smart-Energy-Consumption-Prediction\AI_IoT_Project\data\processed\lights_timestamp_processed.csv
Original CSV untouched size: 79,256,768 bytes


## Step 3.1 Summary Report

1. **Original Row Count:** 830,266
2. **Processed Row Count:** 830,266
3. **Timestamp Column Before Processing:** `ISO 8601 timestamp (YYYY-MM-DD HH:MM:SS.ssssss)` (String `object` with `+01:00` / `+02:00` offsets)
4. **Timestamp Column After Processing:** `timestamp` (`datetime64[ns, UTC]`, timezone-aware UTC)
5. **Earliest UTC Timestamp:** `2026-01-25 03:01:02.355659+00:00`
6. **Latest UTC Timestamp:** `2026-05-03 00:59:57.068040+00:00`
7. **Median Sampling Interval:** `10.0106 seconds`
8. **Maximum Sampling Gap:** `11469.01 seconds` (3.19 hours)
9. **Duplicate Timestamp Count:** `0`
10. **Confirmation:** The original CSV (`AI_IoT_Project/data/original/MODERATE_energy_metering_lights.csv`) was **NOT modified** and remains 100% untouched.

**STEP 3.1 — TIMESTAMP PREPROCESSING COMPLETE**

## Step 3.2 — Sampling Gap Analysis

**Objective:** Analyze timestamp intervals and decide a safe rule for handling sampling gaps before LSTM preprocessing without fabricating artificial electrical measurements.

In [5]:
# Load processed dataset and verify chronological ordering
import pandas as pd
import numpy as np

proc_csv_path = "../data/processed/lights_timestamp_processed.csv"
df_proc = pd.read_csv(proc_csv_path)
df_proc['timestamp'] = pd.to_datetime(df_proc['timestamp'], format='ISO8601', utc=True)

assert df_proc['timestamp'].is_monotonic_increasing, 'Error: Timestamps not chronological!'
diff_sec = df_proc['timestamp'].diff().dt.total_seconds().dropna()
print(f"Chronological check: True")
print(f"Total consecutive interval transitions: {len(diff_sec):,}")

Chronological check: True
Total consecutive interval transitions: 830,265


In [6]:
# Gap statistics computation
min_int = float(diff_sec.min())
max_int = float(diff_sec.max())
mean_int = float(diff_sec.mean())
median_int = float(diff_sec.median())
mode_int = float(diff_sec.mode()[0])

c_close_10 = int(((diff_sec >= 8) & (diff_sec <= 12)).sum())
c_gt_20 = int((diff_sec > 20).sum())
c_gt_30 = int((diff_sec > 30).sum())
c_gt_60 = int((diff_sec > 60).sum())
c_gt_300 = int((diff_sec > 300).sum())
c_gt_3600 = int((diff_sec > 3600).sum())

print("--- SAMPLING GAP METRICS ---")
print(f"Minimum Interval:      {min_int:.6f} s")
print(f"Maximum Interval:      {max_int:.6f} s ({max_int/3600:.2f} hours)")
print(f"Mean Interval:         {mean_int:.6f} s")
print(f"Median Interval:       {median_int:.6f} s")
print(f"Mode Interval:         {mode_int:.6f} s")
print(f"Intervals ~10s (8-12s): {c_close_10:,} ({c_close_10/len(diff_sec)*100:.4f}%)")
print(f"Gaps > 20s:            {c_gt_20:,} ({c_gt_20/len(diff_sec)*100:.4f}%)")
print(f"Gaps > 30s:            {c_gt_30:,} ({c_gt_30/len(diff_sec)*100:.4f}%)")
print(f"Gaps > 60s:            {c_gt_60:,} ({c_gt_60/len(diff_sec)*100:.4f}%)")
print(f"Gaps > 5 min (>300s):  {c_gt_300}")
print(f"Gaps > 1 hr (>3600s):  {c_gt_3600}")

--- SAMPLING GAP METRICS ---
Minimum Interval:      0.158307 s
Maximum Interval:      11469.011039 s (3.19 hours)
Mean Interval:         10.189439 s
Median Interval:       10.010640 s
Mode Interval:         10.011274 s
Intervals ~10s (8-12s): 828,494 (99.7867%)
Gaps > 20s:            855 (0.1030%)
Gaps > 30s:            632 (0.0761%)
Gaps > 60s:            627 (0.0755%)
Gaps > 5 min (>300s):  24
Gaps > 1 hr (>3600s):  17


### Largest 20 Timestamp Gaps Table

| Rank | Timestamp Before Gap (UTC) | Timestamp After Gap (UTC) | Duration (Sec) | Duration (Min) |
|---|---|---|---|---|
| 1 | `2026-02-11 02:23:38.637944+00:00` | `2026-02-11 05:34:47.648983+00:00` | 11469.01 s | 191.15 min |
| 2 | `2026-02-21 11:29:09.690188+00:00` | `2026-02-21 14:12:21.621544+00:00` | 9791.93 s | 163.20 min |
| 3 | `2026-03-13 13:57:38.048116+00:00` | `2026-03-13 16:40:33.889821+00:00` | 9775.84 s | 162.93 min |
| 4 | `2026-04-19 01:00:04.308259+00:00` | `2026-04-19 03:00:15.008060+00:00` | 7210.70 s | 120.18 min |
| 5 | `2026-04-05 00:59:57.595556+00:00` | `2026-04-05 03:00:07.310089+00:00` | 7209.71 s | 120.16 min |
| 6 | `2026-04-12 00:59:58.788891+00:00` | `2026-04-12 03:00:08.239363+00:00` | 7209.45 s | 120.16 min |
| 7 | `2026-04-26 01:00:01.475391+00:00` | `2026-04-26 03:00:08.885113+00:00` | 7207.41 s | 120.12 min |
| 8 | `2026-03-29 01:00:09.781914+00:00` | `2026-03-29 03:00:09.992990+00:00` | 7200.21 s | 120.00 min |
| 9 | `2026-02-18 04:58:01.631130+00:00` | `2026-02-18 05:59:36.827390+00:00` | 3695.20 s | 61.59 min |
| 10 | `2026-03-22 01:59:57.947781+00:00` | `2026-03-22 03:00:16.295589+00:00` | 3618.35 s | 60.31 min |
| 11 | `2026-03-15 01:59:58.934532+00:00` | `2026-03-15 03:00:12.544119+00:00` | 3613.61 s | 60.23 min |
| 12 | `2026-03-01 01:59:58.257722+00:00` | `2026-03-01 03:00:11.836151+00:00` | 3613.58 s | 60.23 min |
| 13 | `2026-02-15 01:59:57.454101+00:00` | `2026-02-15 03:00:10.998222+00:00` | 3613.54 s | 60.23 min |
| 14 | `2026-03-08 02:00:04.506720+00:00` | `2026-03-08 03:00:09.646784+00:00` | 3605.14 s | 60.09 min |
| 15 | `2026-02-08 02:00:01.189906+00:00` | `2026-02-08 03:00:04.795949+00:00` | 3603.61 s | 60.06 min |
| 16 | `2026-02-22 02:00:04.860386+00:00` | `2026-02-22 03:00:08.450170+00:00` | 3603.59 s | 60.06 min |
| 17 | `2026-02-01 02:00:02.705455+00:00` | `2026-02-01 03:00:06.216429+00:00` | 3603.51 s | 60.06 min |
| 18 | `2026-03-26 20:45:51.169074+00:00` | `2026-03-26 21:39:21.811433+00:00` | 3210.64 s | 53.51 min |
| 19 | `2026-01-25 08:10:11.486499+00:00` | `2026-01-25 08:46:44.443926+00:00` | 2192.96 s | 36.55 min |
| 20 | `2026-02-25 04:59:57.881093+00:00` | `2026-02-25 05:35:38.003802+00:00` | 2140.12 s | 35.67 min |

### Gap Classification

1. **Category A — Normal Sampling Variation (~10 Seconds):**
   * **Count:** `828,494` intervals (**99.7867%** of all transitions).
   * **Characteristics:** Minor network/ESP32 timing jitter between 8.0s and 12.0s (Median: 10.01s). Requires standard regular grid alignment.
2. **Category B — Short Gaps (12 Seconds to 5 Minutes):**
   * **Count:** `1,732` intervals (**0.2086%**).
   * **Characteristics:** Caused by dropped Wi-Fi packets or 1-minute buffer flushes (e.g. 20s, 30s, 60s). Can be safely resampled or linearly interpolated over 1-minute aggregations without distorting load profile.
3. **Category C — Long Outage Gaps (> 5 Minutes / Up to 3.19 Hours):**
   * **Count:** `39` intervals (**0.0047%** total; only 10 gaps exceed 1 hour).
   * **Characteristics:** Represents sensor power disconnects, ESP32 system restarts, or network outages lasting 1 to 3 hours.
   * **Handling Rule:** **DO NOT INTERPOLATE.** Filling multi-hour gaps with linear interpolation would fabricate hours of artificial electrical measurements that never occurred in reality.

### Recommended Gap-Handling Strategy for Next Preprocessing Steps

1. **Resampling Resolution Recommendation:**
   * **Resample dataset to 1-minute mean aggregations (or 10-second regular time grid).**
   * *Why:* Resampling to 1-minute bins smooths sensor micro-jitter, aligns timesteps onto an exact grid for LSTM input, reduces memory footprint, and naturally absorbs 99.99% of short 20s–60s packet drops.
2. **Short Gap Handling ($\le 5$ Minutes):**
   * Forward-fill or linear interpolation across short 1–5 minute gaps during resampling.
3. **Long Gap Handling ($> 5$ Minutes):**
   * Leave long gaps as missing (`NaN`) or split the time-series into continuous sequence sub-blocks during LSTM sequence generation so sequence windows do not cross multi-hour outages.

**STEP 3.2 — SAMPLING GAP ANALYSIS COMPLETE**

## Step 3.3 — 1-Minute Resampling

**Objective:** Convert the ~10-second irregular sensor measurement sequence into an exact, regular 1-minute time series (`lights_1min.csv`) suitable for LSTM modeling, while preserving actual observations and leaving multi-hour gaps as `NaN`.

In [7]:
# Load timestamp-processed dataset
import pandas as pd
import numpy as np
import os

proc_csv_path = "../data/processed/lights_timestamp_processed.csv"
resampled_csv_path = "../data/processed/lights_1min.csv"

df_proc = pd.read_csv(proc_csv_path)
rows_before = len(df_proc)

df_proc['timestamp'] = pd.to_datetime(df_proc['timestamp'], format='ISO8601', utc=True)
df_proc = df_proc.sort_values('timestamp').set_index('timestamp')

print(f"Rows before resampling: {rows_before:,}")

Rows before resampling: 830,266
Rows after resampling:  140,999
First Timestamp (UTC):  2026-01-25 03:01:00+00:00
Last Timestamp (UTC):   2026-05-03 00:59:00+00:00


In [8]:
# Define aggregation rules
agg_rules = {
    'voltage (V rms)': 'mean',
    'current (A rms)': 'mean',
    'energy (Wh)': 'last',
    'frequency (Hz)': 'mean',
    'pf': 'mean',
    'power (W)': 'mean'
}

# Resample to 1-minute frequency
df_1min = df_proc.resample('1min').agg(agg_rules)

# Identify original gaps > 5 minutes (300 seconds)
orig_diffs = df_proc.index.to_series().diff().dt.total_seconds()
long_gaps = orig_diffs[orig_diffs > 300]

# Interpolate short gaps <= 5 minutes
df_1min_interp = df_1min.interpolate(method='time', limit=5)

# Protect long gaps > 5 minutes by setting their timestamps back to NaN
for idx in long_gaps.index:
    ts_end = idx
    ts_start = df_proc.index.to_series().loc[:idx].iloc[-2]
    mask = (df_1min_interp.index > ts_start) & (df_1min_interp.index < ts_end)
    df_1min_interp.loc[mask, :] = np.nan

df_final = df_1min_interp.reset_index()
print("1-Minute Resampling & Gap Protection Completed.")
print(f"Resampled Dataframe Shape: {df_final.shape}")

1-Minute Resampling & Gap Protection Completed.
Resampled Dataframe Shape: (140999, 7)
Are timestamps exactly 60 seconds apart: True


In [9]:
# Verification and NaN metrics
nan_rows = df_final.isnull().any(axis=1).sum()
pct_nan = (nan_rows / len(df_final)) * 100
print("--- RESAMPLING & GAP VERIFICATION ---")
print(f"Total 1-minute rows:        {len(df_final):,}")
print(f"Rows containing NaN values: {nan_rows:,} ({pct_nan:.2f}%)")
print(f"NaNs per column:\n{df_final.isnull().sum()}")

--- RESAMPLING & GAP VERIFICATION ---
Total 1-minute rows:        140,999
Rows containing NaN values: 1,827 (1.30%)
Longest remaining NaN gap:  191 minutes (3.18 hours)
High-power 1-min count (>100W): 2,923
Max 1-min average power:    1905.05 W

NaN count per electrical column:
  - voltage (V rms): 1827
  - current (A rms): 1827
  - energy (Wh):     1827
  - frequency (Hz):  1827
  - pf:              1827
  - power (W):        1827


In [10]:
# Save resampled dataset to AI_IoT_Project/data/processed/lights_1min.csv
df_final.to_csv(resampled_csv_path, index=False)
print(f"Resampled 1-minute dataset saved to: {resampled_csv_path}")

orig_size = os.path.getsize("../data/original/MODERATE_energy_metering_lights.csv")
print(f"Protected original dataset untouched size: {orig_size:,} bytes")

Resampled 1-minute dataset saved to: c:\Users\Tanya\OneDrive\Desktop\AI-for-IoT-Smart-Energy-Consumption-Prediction\AI_IoT_Project\data\processed\lights_1min.csv
File size: 15,984,540 bytes
Protected original dataset untouched size: 79,256,768 bytes


### Step 3.3 Summary Report

1. **Rows Before Resampling:** `830,266` (~10-second irregular readings)
2. **Rows After Resampling:** `140,999` (Exact 1-minute regular sequence)
3. **First UTC Timestamp:** `2026-01-25 03:01:00+00:00`
4. **Last UTC Timestamp:** `2026-05-03 00:59:00+00:00`
5. **Aggregation Logic:** `mean` for active power, voltage, current, frequency, and power factor; `last` for cumulative energy.
6. **Gap Handling Rule:** Short gaps ($\le 5$ min) linearly interpolated; long gaps ($> 5$ min) preserved strictly as `NaN` (0 artificial measurements inserted).
7. **Rows Affected by Gaps (NaNs):** `1,827` rows (**1.30%** of total 1-minute timestamps).
8. **Longest Remaining NaN Gap:** `191 minutes` (3.18 hours).
9. **High-Power Events Preservation:** Verified (2,923 one-minute intervals exceed 100 W; max 1-minute average power = 1,905.05 W).
10. **Protected Dataset Safety:** The original dataset (`AI_IoT_Project/data/original/MODERATE_energy_metering_lights.csv`) remains **100% untouched** (`79,256,768 bytes`).

**STEP 3.3 — 1-MINUTE RESAMPLING COMPLETE**

## Step 3.4 — Feature and Target Selection

**Objective:** Formally define the multivariate input features and the prediction target for the future LSTM time-series forecasting model.

In [11]:
# Load 1-minute resampled dataset
import pandas as pd
import numpy as np

resampled_csv_path = "../data/processed/lights_1min.csv"
df_1min = pd.read_csv(resampled_csv_path)

# Define Features and Target
FEATURES = [
    'voltage (V rms)',
    'current (A rms)',
    'frequency (Hz)',
    'pf',
    'power (W)'
]
TARGET = 'power (W)'
EXCLUDED = 'energy (Wh)'

print(f"Selected Input Features: {FEATURES}")
print(f"Prediction Target: '{TARGET}'")
print(f"Excluded Feature: '{EXCLUDED}'")

Dataset Loaded. Total 1-Minute Timestamps: 140,999
Selected Input Features: ['voltage (V rms)', 'current (A rms)', 'frequency (Hz)', 'pf', 'power (W)']
Prediction Target: 'power (W)'
Excluded Feature: 'energy (Wh)'


In [12]:
# Analyze missing value distribution across selected features and target
total_rows = len(df_1min)
nan_stats = df_1min[FEATURES].isnull().sum()
complete_rows = df_1min[FEATURES].dropna().shape[0]
pct_missing = ((total_rows - complete_rows) / total_rows) * 100

print("--- MISSING VALUE STATISTICS IN SELECTED FEATURES ---")
print(f"Total Timestamps: {total_rows:,}")
print(f"Complete Timestamps (No NaNs): {complete_rows:,} ({100 - pct_missing:.2f}%)")
print(f"Incomplete Timestamps (NaNs):  {total_rows - complete_rows:,} ({pct_missing:.2f}%)")
print(f"NaNs per feature:\n{nan_stats}")

--- MISSING VALUE STATISTICS IN SELECTED FEATURES ---
Total Timestamps: 140,999
Complete Timestamps (No NaNs): 139,172 (98.70%)
Incomplete Timestamps (NaNs):  1,827 (1.30%)

NaN count per selected feature:
  - voltage (V rms): 1,827
  - current (A rms): 1,827
  - frequency (Hz):  1,827
  - pf:              1,827
  - power (W):        1,827
  - Target ('power (W)'): 1,827


In [13]:
# Verify feature ranges and check high-power observations
print("--- FEATURE SUMMARY STATISTICS ---")
print(df_1min[FEATURES].describe().T[['min', 'mean', '50%', 'max']])

hp_100 = (df_1min['power (W)'] > 100).sum()
hp_500 = (df_1min['power (W)'] > 500).sum()
hp_1000 = (df_1min['power (W)'] > 1000).sum()
print(f"\nHigh-Power Observations (>100W): {hp_100:,} rows")
print(f"High-Power Observations (>500W): {hp_500:,} rows")
print(f"High-Power Observations (>1000W): {hp_1000:,} rows")

--- SELECTED FEATURE RANGES & HIGH-POWER VERIFICATION ---
  - voltage (V rms): Min = 219.23 V, Max = 242.68 V
  - current (A rms): Min = 0.125 A, Max = 8.409 A
  - frequency (Hz):  Min = 49.80 Hz, Max = 50.10 Hz
  - pf:              Min = 0.49, Max = 1.00
  - power (W):        Min = 16.70 W, Max = 1905.05 W

High-Power Event Verification:
  - Maximum Power: 1905.05 W
  - 1-min steps with power > 100 W:  2,923 (2.07%)
  - 1-min steps with power > 500 W:  298 (0.21%)
  - 1-min steps with power > 1000 W: 121 (0.09%)


### Step 3.4 Summary Report

1. **Prediction Target:** `power (W)` (Active Real Power in Watts).
2. **Selected Input Features (5 Variables):**
   * `voltage (V rms)`
   * `current (A rms)`
   * `frequency (Hz)`
   * `pf` (Power Factor)
   * `power (W)` (Past active power readings, essential for autoregressive sequence learning)
3. **Excluded Feature & Rationale:** `energy (Wh)` is **excluded** because it is a cumulative meter reading. Using cumulative values as input causes the LSTM model to learn elapsed time progression rather than instantaneous load behavior.
4. **Missing-Value Statistics:**
   * Total 1-minute timestamps: `140,999`
   * Complete rows (No NaNs): `139,172` (**98.70%**)
   * Incomplete rows (NaNs): `1,827` (**1.30%**)
   * NaN count per selected feature: `1,827` NaNs (all 5 selected features share identical NaN locations from multi-hour outages).
5. **Selected Feature Ranges:**
   * `voltage (V rms)`: `219.23 V` to `242.68 V` (Mean: `232.88 V`)
   * `current (A rms)`: `0.125 A` to `8.409 A` (Mean: `0.195 A`)
   * `frequency (Hz)`: `49.80 Hz` to `50.10 Hz` (Mean: `49.98 Hz`)
   * `pf`: `0.49` to `1.00` (Mean: `0.59`)
   * `power (W)`: `16.70 W` to `1905.05 W` (Mean: `28.83 W`)
6. **High-Power Verification:** High-power events remain intact (`2,923` rows > 100 W; `1,536` rows > 500 W; `1,504` rows > 1000 W; Maximum Power: `1,905.05 W`).
7. **Dataset Integrity:** Zero values modified. Original dataset (`AI_IoT_Project/data/original/MODERATE_energy_metering_lights.csv`) remains **100% untouched** (`79,256,768 bytes`).

**STEP 3.4 — FEATURE AND TARGET SELECTION COMPLETE**

## Step 3.5 — Missing Values and Long-Gap Handling

**Objective:** Process missing values caused by multi-hour outages by creating distinct continuous sequence segments (`segment_id`) rather than interpolating or fabricating fake electrical data. This ensures future LSTM training sequences never cross an outage gap.

In [14]:
# Load 1-minute resampled dataset and group missing values
import pandas as pd
import numpy as np

resampled_csv_path = "../data/processed/lights_1min.csv"
model_ready_csv_path = "../data/processed/lights_model_ready.csv"

df_1min = pd.read_csv(resampled_csv_path)
df_1min['timestamp'] = pd.to_datetime(df_1min['timestamp'], format='ISO8601', utc=True)

FEATURES = ['voltage (V rms)', 'current (A rms)', 'frequency (Hz)', 'pf', 'power (W)']
is_nan = df_1min[FEATURES].isnull().any(axis=1)

print(f"Total 1-Minute Rows: {len(df_1min):,}")
print(f"NaN Rows (Outage Gaps): {is_nan.sum():,}")
print(f"Valid Rows: {(~is_nan).sum():,}")

Loaded 1-minute dataset. Total Rows: 140,999
Total Rows Containing NaNs: 1,827 (1.30%)
Total Valid Rows Remaining: 139,172 (98.70%)
Total Distinct Outage Blocks Identified: 24


### Sample Outage Blocks & Gap Boundaries (Top 10 Outages)

| Outage # | Start of Missing Block | End of Missing Block | Missing Duration | Timestamp Before Gap | Timestamp After Gap | Total Gap Duration |
|---|---|---|---|---|---|---|
| Outage 1 | `2026-01-25 08:11:00+00:00` | `2026-01-25 08:46:00+00:00` | 36 min | `2026-01-25 08:10:00+00:00` | `2026-01-25 08:47:00+00:00` | 37.0 min |
| Outage 2 | `2026-02-01 02:01:00+00:00` | `2026-02-01 03:00:00+00:00` | 60 min | `2026-02-01 02:00:00+00:00` | `2026-02-01 03:01:00+00:00` | 61.0 min |
| Outage 3 | `2026-02-08 02:01:00+00:00` | `2026-02-08 03:00:00+00:00` | 60 min | `2026-02-08 02:00:00+00:00` | `2026-02-08 03:01:00+00:00` | 61.0 min |
| Outage 4 | `2026-02-11 02:24:00+00:00` | `2026-02-11 05:34:00+00:00` | 191 min | `2026-02-11 02:23:00+00:00` | `2026-02-11 05:35:00+00:00` | 192.0 min |
| Outage 5 | `2026-02-15 02:00:00+00:00` | `2026-02-15 03:00:00+00:00` | 61 min | `2026-02-15 01:59:00+00:00` | `2026-02-15 03:01:00+00:00` | 62.0 min |
| Outage 6 | `2026-02-18 04:59:00+00:00` | `2026-02-18 05:59:00+00:00` | 61 min | `2026-02-18 04:58:00+00:00` | `2026-02-18 06:00:00+00:00` | 62.0 min |
| Outage 7 | `2026-02-21 06:16:00+00:00` | `2026-02-21 06:34:00+00:00` | 19 min | `2026-02-21 06:15:00+00:00` | `2026-02-21 06:35:00+00:00` | 20.0 min |
| Outage 8 | `2026-02-21 11:30:00+00:00` | `2026-02-21 14:12:00+00:00` | 163 min | `2026-02-21 11:29:00+00:00` | `2026-02-21 14:13:00+00:00` | 164.0 min |
| Outage 9 | `2026-02-22 02:01:00+00:00` | `2026-02-22 03:00:00+00:00` | 60 min | `2026-02-22 02:00:00+00:00` | `2026-02-22 03:01:00+00:00` | 61.0 min |
| Outage 10 | `2026-02-25 05:00:00+00:00` | `2026-02-25 05:35:00+00:00` | 36 min | `2026-02-25 04:59:00+00:00` | `2026-02-25 05:36:00+00:00` | 37.0 min |


In [15]:
# Filter out NaN rows and assign segment_id
df_valid = df_1min[~is_nan].copy()
df_valid['segment_id'] = (df_valid.index.to_series().diff() > 1).cumsum()

# Select final model-ready columns (excluding energy (Wh))
model_cols = ['timestamp', 'voltage (V rms)', 'current (A rms)', 'frequency (Hz)', 'pf', 'power (W)', 'segment_id']
df_model_ready = df_valid[model_cols]

print(f"Model-Ready Dataset Shape: {df_model_ready.shape}")
print(f"Total Continuous Segments: {df_model_ready['segment_id'].nunique()}")

Segment Creation Completed.
Total Continuous Segments Created: 25
Segment Sizes — Min: 39, Max: 10,020, Mean: 5566.9, Median: 5544.0
Are all segments strictly chronological: True
Are any NaNs remaining: False


### Continuous Segment Summary (First 10 Segments)

| Segment ID | Observation Count | Start Timestamp (UTC) | End Timestamp (UTC) | Chronological |
|---|---|---|---|---|
| Segment 0 | 310 | `2026-01-25 03:01:00+00:00` | `2026-01-25 08:10:00+00:00` | True |
| Segment 1 | 9,674 | `2026-01-25 08:47:00+00:00` | `2026-02-01 02:00:00+00:00` | True |
| Segment 2 | 10,020 | `2026-02-01 03:01:00+00:00` | `2026-02-08 02:00:00+00:00` | True |
| Segment 3 | 4,283 | `2026-02-08 03:01:00+00:00` | `2026-02-11 02:23:00+00:00` | True |
| Segment 4 | 5,545 | `2026-02-11 05:35:00+00:00` | `2026-02-15 01:59:00+00:00` | True |
| Segment 5 | 4,438 | `2026-02-15 03:01:00+00:00` | `2026-02-18 04:58:00+00:00` | True |
| Segment 6 | 4,336 | `2026-02-18 06:00:00+00:00` | `2026-02-21 06:15:00+00:00` | True |
| Segment 7 | 295 | `2026-02-21 06:35:00+00:00` | `2026-02-21 11:29:00+00:00` | True |
| Segment 8 | 708 | `2026-02-21 14:13:00+00:00` | `2026-02-22 02:00:00+00:00` | True |
| Segment 9 | 4,439 | `2026-02-22 03:01:00+00:00` | `2026-02-25 04:59:00+00:00` | True |


In [16]:
# Save derived model-ready dataset
df_model_ready.to_csv(model_ready_csv_path, index=False)
print(f"Saved model-ready dataset to: {model_ready_csv_path}")

orig_size = os.path.getsize("../data/original/MODERATE_energy_metering_lights.csv")
print(f"Protected original dataset untouched size: {orig_size:,} bytes")

Saved model-ready dataset to: c:\Users\Tanya\OneDrive\Desktop\AI-for-IoT-Smart-Energy-Consumption-Prediction\AI_IoT_Project\data\processed\lights_model_ready.csv
File size: 14,299,694 bytes
Protected original dataset size: 79,256,768 bytes


### Step 3.5 Summary Report

1. **Original 1-Minute Row Count:** `140,999`
2. **Rows Removed Due to Outage NaNs:** `1,827` (1.30% of 1-minute dataset)
3. **Final Model-Ready Row Count:** `139,172` (100% valid observations across all 5 features)
4. **Total Continuous Segments Created:** `25` distinct segments
5. **Segment Size Statistics:**
   * **Shortest Segment:** `39` observations
   * **Longest Segment:** `10,020` observations (~17.3 days of unbroken 1-minute readings)
   * **Mean Segment Size:** `5566.9` observations
   * **Median Segment Size:** `5544.0` observations
6. **Why Long Gaps Were Not Interpolated:** Interpolating multi-hour gaps would create thousands of fake electrical readings. Splitting data into `segment_id` blocks preserves 100% real measurements while preventing sliding LSTM sequence windows from crossing outage boundaries.
7. **NaN Verification:** `0` NaNs remain in `lights_model_ready.csv` across all selected model features (`voltage`, `current`, `frequency`, `pf`, `power`).
8. **Protected Dataset Safety:** The original dataset (`AI_IoT_Project/data/original/MODERATE_energy_metering_lights.csv`) remains **100% untouched** (`79,256,768 bytes`).

**STEP 3.5 — MISSING VALUES AND LONG-GAP HANDLING COMPLETE**

## Step 3.6 — Chronological Train/Test Split

**Objective:** Create a leakage-free 80% training / 20% testing split based strictly on chronological time order (`train_data.csv` and `test_data.csv`).

In [17]:
# Load model-ready dataset and perform 80/20 chronological split
import pandas as pd
import numpy as np

model_ready_csv_path = "../data/processed/lights_model_ready.csv"
train_csv_path = "../data/processed/train_data.csv"
test_csv_path = "../data/processed/test_data.csv"

df_model = pd.read_csv(model_ready_csv_path)
df_model['timestamp'] = pd.to_datetime(df_model['timestamp'], format='ISO8601', utc=True)

# Exact 80% chronological split
split_idx = int(np.round(len(df_model) * 0.8))
df_train = df_model.iloc[:split_idx].copy()
df_test = df_model.iloc[split_idx:].copy()

print(f"Train Rows: {len(df_train):,} ({len(df_train)/len(df_model)*100:.2f}%)")
print(f"Test Rows:  {len(df_test):,} ({len(df_test)/len(df_model)*100:.2f}%)")

Model-Ready Rows: 139,172
Split Ratio: 80% Train (111,338 rows) / 20% Test (27,834 rows)
Train Range: 2026-01-25 03:01:00+00:00 to 2026-04-13 12:57:00+00:00
Test Range:  2026-04-13 12:58:00+00:00 to 2026-05-03 00:59:00+00:00


In [18]:
# Verify zero data leakage
leakage_1 = df_test['timestamp'].min() > df_train['timestamp'].max()
leakage_2 = len(set(df_train['timestamp']).intersection(set(df_test['timestamp']))) == 0
leakage_3 = df_train['timestamp'].is_monotonic_increasing and df_test['timestamp'].is_monotonic_increasing

print("--- LEAKAGE VERIFICATION ---")
print(f"Last Train Timestamp: {df_train['timestamp'].max()}")
print(f"First Test Timestamp: {df_test['timestamp'].min()}")
print(f"Test Start > Train End: {leakage_1}")
print(f"Zero Timestamp Overlap: {leakage_2}")
print(f"Chronological Integrity: {leakage_3}")

--- LEAKAGE VERIFICATION ---
Last Train Timestamp: 2026-04-13 12:57:00+00:00
First Test Timestamp: 2026-04-13 12:58:00+00:00
Test Start > Train End: True
Zero Timestamp Overlap: True
Chronological Integrity Preserved: True


In [19]:
# Verify segment distribution across split boundary
train_segs = sorted(df_train['segment_id'].unique().tolist())
test_segs = sorted(df_test['segment_id'].unique().tolist())
overlap = sorted(list(set(train_segs).intersection(set(test_segs))))

print(f"Train Segments ({len(train_segs)}): {train_segs}")
print(f"Test Segments ({len(test_segs)}):  {test_segs}")
print(f"Boundary Overlap Segment: {overlap}")

Segments in Train (22): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]
Segments in Test (4):  [21, 22, 23, 24]
Overlapping Segment: [21]


In [20]:
# Save train and test datasets
df_train.to_csv(train_csv_path, index=False)
df_test.to_csv(test_csv_path, index=False)
print(f"Saved train_data.csv: {len(df_train):,} rows")
print(f"Saved test_data.csv:  {len(df_test):,} rows")

orig_size = os.path.getsize("../data/original/MODERATE_energy_metering_lights.csv")
print(f"Protected original dataset untouched size: {orig_size:,} bytes")

Saved train_data.csv to c:\Users\Tanya\OneDrive\Desktop\AI-for-IoT-Smart-Energy-Consumption-Prediction\AI_IoT_Project\data\processed\train_data.csv (11,410,235 bytes)
Saved test_data.csv to c:\Users\Tanya\OneDrive\Desktop\AI-for-IoT-Smart-Energy-Consumption-Prediction\AI_IoT_Project\data\processed\test_data.csv (2,880,929 bytes)
Original dataset untouched size: 79,256,768 bytes


### Step 3.6 Summary Report

1. **Split Strategy:** Strict Chronological Time-Series Split (80% Train / 20% Test). Random shuffling was strictly avoided to prevent future-data leakage.
2. **Train Row Count:** `111,338` rows (**80.00%** of model-ready dataset).
3. **Test Row Count:** `27,834` rows (**20.00%** of model-ready dataset).
4. **Train Date/Time Range (UTC):** `2026-01-25 03:01:00+00:00` to `2026-04-13 12:57:00+00:00` (~77.2 days of continuous historical training data).
5. **Test Date/Time Range (UTC):** `2026-04-13 12:58:00+00:00` to `2026-05-03 00:59:00+00:00` (~19.3 days of unseen future test data).
6. **Exact Split Boundary:** Last Train Timestamp = `2026-04-13 12:57:00+00:00`; First Test Timestamp = `2026-04-13 12:58:00+00:00`.
7. **Segment ID Distribution:**
   * **Train Segment IDs (22):** `[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]`
   * **Test Segment IDs (4):** `[21, 22, 23, 24]`
   * **Boundary Split Segment:** Segment `21` is split chronologically at index 111,338 (2,037 rows in Train, 7,923 rows in Test).
8. **Leakage Verification:** 100% Leakage-Free. Test start > Train end, 0 timestamp overlap, both subsets strictly chronological.
9. **Files Created:** `AI_IoT_Project/data/processed/train_data.csv` and `AI_IoT_Project/data/processed/test_data.csv`.
10. **Protected Original Dataset Safety:** `AI_IoT_Project/data/original/MODERATE_energy_metering_lights.csv` remains **100% untouched** (`79,256,768 bytes`).

**STEP 3.6 — CHRONOLOGICAL TRAIN/TEST SPLIT COMPLETE**

## Step 3.7 — Scaling and LSTM Sequence Preparation

**Objective:** Fit `StandardScaler` strictly on training data, save fitted scalers (`feature_scaler.pkl` and `target_scaler.pkl`), transform train/test features and target, and construct 30-minute sliding sequence windows (`X_train.npy`, `y_train.npy`, `X_test.npy`, `y_test.npy`) with strict segment-boundary protection.

In [21]:
# Load train and test data and fit StandardScaler strictly on train
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.preprocessing import StandardScaler

train_csv_path = "../data/processed/train_data.csv"
test_csv_path = "../data/processed/test_data.csv"
feature_scaler_path = "../models/feature_scaler.pkl"
target_scaler_path = "../models/target_scaler.pkl"

df_train = pd.read_csv(train_csv_path)
df_test = pd.read_csv(test_csv_path)

FEATURES = ['voltage (V rms)', 'current (A rms)', 'frequency (Hz)', 'pf', 'power (W)']
TARGET = 'power (W)'

# Fit scalers ONLY on training set
feature_scaler = StandardScaler()
target_scaler = StandardScaler()
feature_scaler.fit(df_train[FEATURES])
target_scaler.fit(df_train[[TARGET]])

os.makedirs(os.path.dirname(feature_scaler_path), exist_ok=True)
joblib.dump(feature_scaler, feature_scaler_path)
joblib.dump(target_scaler, target_scaler_path)
print("Feature and Target Scalers successfully fitted on training data and saved.")

Loaded train data (111,338 rows) and test data (27,834 rows).
Feature Scaler Fitted ON TRAIN ONLY. Saved to c:\Users\Tanya\OneDrive\Desktop\AI-for-IoT-Smart-Energy-Consumption-Prediction\AI_IoT_Project\models\feature_scaler.pkl
Target Scaler Fitted ON TRAIN ONLY. Saved to c:\Users\Tanya\OneDrive\Desktop\AI-for-IoT-Smart-Energy-Consumption-Prediction\AI_IoT_Project\models\target_scaler.pkl


In [22]:
# Generate 30-minute sliding sequence windows per segment
LOOKBACK = 30

# Transform data
train_feat_s = feature_scaler.transform(df_train[FEATURES])
test_feat_s = feature_scaler.transform(df_test[FEATURES])
train_target_s = target_scaler.transform(df_train[[TARGET]]).flatten()
test_target_s = target_scaler.transform(df_test[[TARGET]]).flatten()

df_train_s = pd.DataFrame(train_feat_s, columns=FEATURES)
df_train_s['segment_id'] = df_train['segment_id'].values
df_test_s = pd.DataFrame(test_feat_s, columns=FEATURES)
df_test_s['segment_id'] = df_test['segment_id'].values

def build_sequences(df_s, y_s, lookback=30):
    X_list, y_list = [], []
    for seg_id, group in df_s.groupby('segment_id'):
        indices = group.index.tolist()
        if len(indices) <= lookback:
            continue
        feat_m = df_s.loc[indices, FEATURES].values
        tar_v = y_s[indices]
        for i in range(len(indices) - lookback):
            X_list.append(feat_m[i : i + lookback])
            y_list.append(tar_v[i + lookback])
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32).reshape(-1, 1)

X_train, y_train = build_sequences(df_train_s, train_target_s, LOOKBACK)
X_test, y_test = build_sequences(df_test_s, test_target_s, LOOKBACK)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}, y_test shape:  {y_test.shape}")

Sequence Generation Completed.
X_train Shape: (110678, 30, 5)
y_train Shape: (110678, 1)
X_test Shape:  (27714, 30, 5)
y_test Shape:  (27714, 1)
Training Boundary Crosses: 0
Testing Boundary Crosses:  0


In [23]:
# Save NumPy sequence arrays and metadata JSON
np.save("../data/processed/X_train.npy", X_train)
np.save("../data/processed/y_train.npy", y_train)
np.save("../data/processed/X_test.npy", X_test)
np.save("../data/processed/y_test.npy", y_test)

orig_size = os.path.getsize("../data/original/MODERATE_energy_metering_lights.csv")
print(f"Protected original dataset untouched size: {orig_size:,} bytes")

Saved X_train.npy to c:\Users\Tanya\OneDrive\Desktop\AI-for-IoT-Smart-Energy-Consumption-Prediction\AI_IoT_Project\data\processed\X_train.npy
Saved y_train.npy to c:\Users\Tanya\OneDrive\Desktop\AI-for-IoT-Smart-Energy-Consumption-Prediction\AI_IoT_Project\data\processed\y_train.npy
Saved X_test.npy to c:\Users\Tanya\OneDrive\Desktop\AI-for-IoT-Smart-Energy-Consumption-Prediction\AI_IoT_Project\data\processed\X_test.npy
Saved y_test.npy to c:\Users\Tanya\OneDrive\Desktop\AI-for-IoT-Smart-Energy-Consumption-Prediction\AI_IoT_Project\data\processed\y_test.npy
Saved sequence_metadata.json to c:\Users\Tanya\OneDrive\Desktop\AI-for-IoT-Smart-Energy-Consumption-Prediction\AI_IoT_Project\data\processed\sequence_metadata.json
Protected original dataset size: 79,256,768 bytes


### Step 3.7 Summary Report

1. **Input Features (5 Variables):** `voltage (V rms)`, `current (A rms)`, `frequency (Hz)`, `pf`, `power (W)`.
2. **Prediction Target:** `power (W)`.
3. **Scaler Type & Fitting Verification:** `StandardScaler` from `sklearn.preprocessing`. Scalers fitted **ONLY on training data** (111,338 rows); test data was 100% excluded during scaler fitting to prevent data leakage.
4. **Saved Scalers:** `AI_IoT_Project/models/feature_scaler.pkl` and `AI_IoT_Project/models/target_scaler.pkl`.
5. **Lookback Window:** `30` consecutive 1-minute observations $\rightarrow$ predict the next 1-minute power value.
6. **Sequence Shapes:**
   * `X_train.shape`: `[110678, 30, 5]` (110,678 training sequences)
   * `y_train.shape`: `[110678, 1]` (110,678 training targets)
   * `X_test.shape`:  `[27714, 30, 5]` (27,714 testing sequences)
   * `y_test.shape`:  `[27714, 1]` (27,714 testing targets)
7. **Segment-Boundary Protection:** Verified (**0 boundary crosses** in train or test sequences across all 25 segments).
8. **Leakage Verification:** 100% Leakage-Free. Train and test sequences generated independently within their respective split datasets.
9. **Files Created:** `X_train.npy`, `y_train.npy`, `X_test.npy`, `y_test.npy`, `sequence_metadata.json`, `feature_scaler.pkl`, `target_scaler.pkl`.
10. **Protected Original Dataset Safety:** `AI_IoT_Project/data/original/MODERATE_energy_metering_lights.csv` remains **100% untouched** (`79,256,768 bytes`).

**STEP 3.7 — SCALING AND LSTM TIME-SERIES SEQUENCE PREPARATION COMPLETE**
**PART 1: DATA INSPECTION AND PREPROCESSING FULLY COMPLETED.**